In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('ecommerce_customer_churn_messy.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.dtypes

In [ ]:
df.describe()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

In [ ]:
df.isnull().mean() * 100

In [ ]:
df.duplicated().sum()

In [ ]:
df["churn"].value_counts()

In [ ]:
df["churn"].value_counts(normalize=True) * 100

In [ ]:
# ==========================================
# 1. Remove exact duplicates
# ==========================================

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)

In [ ]:
# ==========================================
# 2. Convert fake missing values to NaN
# ==========================================

missing_tokens = [
    "",
    "na",
    "n/a",
    "nan",
    "null",
    "none",
    "unknown",
    "?"
]

for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = (
        df[col]
        .astype("string")
        .str.strip()
        .str.lower()
        .replace(
            {token: np.nan for token in missing_tokens}
        )
    )

In [ ]:
# ==========================================
# 3. Clean gender
# ==========================================

gender_map = {
    "female": "Female",
    "f": "Female",
    "male": "Male",
    "m": "Male"
}

df["gender"] = df["gender"].map(gender_map)

In [ ]:
# ==========================================
# 4. Clean membership_type
# ==========================================

df["membership_type"] = df["membership_type"].replace({
    "basic": "Basic",
    "premium": "Premium",
    "gold": "Gold",
    "platinum": "Platinum"
})

In [ ]:
df["membership_type"] = (
    df["membership_type"]
    .str.strip()
    .str.lower()
    .replace({
        "basic": "Basic",
        "premium": "Premium",
        "gold": "Gold",
        "platinum": "Platinum"
    })
)

In [ ]:
df["membership_type"].value_counts()

In [ ]:
def parse_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().lower()

    if value in ["", "na", "n/a", "unknown", "?", "-", "null"]:
        return np.nan

    value = value.replace(",", "")
    value = value.replace("$", "")
    value = value.replace("usd", "")

    if value.endswith("k"):
        return float(value[:-1]) * 1000

    return float(value)

In [ ]:
numeric_columns = [
    "customer_age",
    "annual_income",
    "account_balance",
    "total_spending",
    "average_order_value",
    "number_of_orders",
    "days_since_last_order",
    "website_visits",
    "support_tickets",
    "discount_usage",
    "refund_amount",
    "lifetime_value"
]

In [ ]:
for col in numeric_columns:
    df[col] = df[col].apply(parse_number)

In [ ]:
df[numeric_columns].dtypes

In [ ]:
df.loc[
    ~df["customer_age"].between(18, 100),
    "customer_age"
] = np.nan

In [ ]:
non_negative_cols = [
    "annual_income",
    "total_spending",
    "average_order_value",
    "number_of_orders",
    "days_since_last_order",
    "website_visits",
    "support_tickets",
    "discount_usage",
    "refund_amount",
    "lifetime_value"
]

for col in non_negative_cols:
    df.loc[df[col] < 0, col] = np.nan

In [ ]:
date_columns = [
    "signup_date",
    "subscription_start",
    "subscription_end",
    "last_purchase_date",
    "last_login",
    "cancellation_date"
]

for col in date_columns:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        errors="coerce"
    )

In [ ]:
reference_date = pd.Timestamp("2025-07-01")

df["days_since_login"] = (
    reference_date - df["last_login"]
).dt.days

In [ ]:
df["days_since_login"]

In [ ]:
leakage_columns = [
    "cancellation_reason",
    "cancellation_date",
    "final_account_status"
]

df = df.drop(columns=leakage_columns)

In [ ]:
df = df.drop(columns=["customer_id"])

In [ ]:
print("Numeric features:")
print(numeric_features)

print("\nCategorical features:")
print(categorical_features)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [ ]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

In [ ]:
# Create model data before splitting
datetime_columns = df.select_dtypes(
    include=["datetime64[ns]", "datetimetz"]
).columns.tolist()

model_df = df.drop(columns=datetime_columns, errors="ignore")

X = model_df.drop(columns=["churn"])
y = model_df["churn"]

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# Remove datetime columns before model training
datetime_columns = df.select_dtypes(
    include=["datetime", "datetimetz"]
).columns.tolist()

model_df = df.drop(columns=datetime_columns, errors="ignore")

# Define features and target before using them
X = model_df.drop(columns=["churn"])
y = model_df["churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

numeric_features = X_train.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

logistic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

y_pred_logistic = logistic_model.predict(X_test)

print("Accuracy :", accuracy_score(y_test, y_pred_logistic))
print("Precision:", precision_score(y_test, y_pred_logistic, zero_division=0))
print("Recall   :", recall_score(y_test, y_pred_logistic, zero_division=0))
print("F1 Score :", f1_score(y_test, y_pred_logistic, zero_division=0))
print(classification_report(y_test, y_pred_logistic, zero_division=0))

In [ ]:
y_pred_logistic = logistic_model.predict(X_test)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

In [ ]:
print("Accuracy :", accuracy_score(y_test, y_pred_logistic))
print("Precision:", precision_score(y_test, y_pred_logistic))
print("Recall   :", recall_score(y_test, y_pred_logistic))
print("F1 Score :", f1_score(y_test, y_pred_logistic))
print(classification_report(y_test, y_pred_logistic))